# Step 23 — ADP-B Cloud Training (Gemma-2-2b-it QLoRA)
**Phase:** 4.1 — Cloud Retraining
**Spec authority:** SPEC-400 §3.3, SPEC-300; REQ-400-060–075
**Platform:** Lightning.ai A10G (24 GB VRAM)
**Base model:** `google/gemma-2-2b-it`
**Prerequisites:** Step 22 complete
**Output adapter:** `finetuning/adp_b_safety/adp_b_cloud_final/` → pushed to HF Hub

---

## A10G vs RTX 3070 differences

| Parameter | Local (Step 16) | Cloud (Step 23) | Reason |
|-----------|----------------|----------------|--------|
| Quantisation | None (bf16) | **NF4 4-bit** | Frees VRAM for larger effective batch |
| `BATCH_SIZE` | 4 | **8** | A10G headroom |
| `GRAD_ACCUM` | 4 | **2** | Effective batch 16 maintained |
| `MAX_SEQ_LENGTH` | 512 | **768** | Extended output schema (routing_mode, confidence, pubmed_eligible) |
| `NUM_EPOCHS` | 40 + early stop | **30** + early stop | Larger dataset converges faster |
| `LORA_TARGETS` | q/k/v/o + MLP | **same** | No change — MLP targets necessary for format override |

Loss target unchanged: **< 0.30** (narrow classification space).


In [ ]:
# ── Cell 0: Environment ──────────────────────────────────────────────────────
import os, json, gc, random
from pathlib import Path
from collections import Counter, defaultdict

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HF Hub authenticated.")

import torch
print(f"Device : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")


In [ ]:
# ── Cell 1: Configuration ────────────────────────────────────────────────────
BASE_MODEL_ID  = "google/gemma-2-2b-it"
HF_OUTPUT_REPO = "equinox013/nikko-adp-b"   # update to your repo

BASE_DIR       = Path("/teamspace/studios/this_studio/nikko-companion")
FINETUNING     = BASE_DIR / "finetuning"
TRAIN_DATA     = FINETUNING / "adp_b_safety" / "data" / "adp_b_cloud_train.jsonl"
CHECKPOINT_DIR = FINETUNING / "adp_b_safety" / "cloud_checkpoints"
OUTPUT_DIR     = FINETUNING / "adp_b_safety" / "adp_b_cloud_final"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DATA.exists(), f"Missing: {TRAIN_DATA} — run Step 22 first."

NUM_EPOCHS     = 30
BATCH_SIZE     = 8
GRAD_ACCUM     = 2       # effective batch = 16
MAX_SEQ_LENGTH = 768     # accommodates extended output schema + full system preamble
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 0.01

LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "gate_proj", "up_proj", "down_proj"]

print(f"Base model     : {BASE_MODEL_ID}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}  |  MaxSeqLen: {MAX_SEQ_LENGTH}")
print(f"Output         : {OUTPUT_DIR}")


In [ ]:
# ── Cell 2: Dataset ──────────────────────────────────────────────────────────
from datasets import Dataset
from transformers import AutoTokenizer

random.seed(42)

raw_records = [json.loads(l) for l in TRAIN_DATA.read_text().splitlines() if l.strip()]
print(f"Loaded {len(raw_records)} records.")

# Distribution check before training — safety data must be class-balanced.
is_crisis_dist = Counter(json.loads(r["output"]).get("is_crisis", False) for r in raw_records)
routing_dist   = Counter(json.loads(r["output"]).get("routing_mode", "unknown") for r in raw_records)
print(f"is_crisis distribution : {dict(is_crisis_dist)}")
print(f"routing_mode dist      : {dict(routing_dist)}")

if is_crisis_dist.get(True, 0) / len(raw_records) < 0.25:
    print("WARN: Less than 25% crisis-positive examples — model may underclassify crisis.")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def fmt(r):
    msgs = [{"role": "user", "content": r["instruction"]},
            {"role": "assistant", "content": r["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

# Stratified split by is_crisis label
buckets = defaultdict(list)
for r in raw_records:
    k = json.loads(r["output"]).get("is_crisis", False)
    buckets[k].append(r)

train_recs, eval_recs = [], []
for _, items in buckets.items():
    random.shuffle(items)
    n_eval = max(1, int(len(items) * 0.10))
    eval_recs.extend(items[:n_eval])
    train_recs.extend(items[n_eval:])

random.shuffle(train_recs)
random.shuffle(eval_recs)
train_data = Dataset.from_list([fmt(r) for r in train_recs])
eval_data  = Dataset.from_list([fmt(r) for r in eval_recs])
print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")


In [ ]:
# ── Cell 3: Model load (NF4 QLoRA) ───────────────────────────────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

gc.collect(); torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT, bias="none", target_modules=LORA_TARGETS,
    use_rslora=True,
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f"VRAM after load+LoRA: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


In [ ]:
# ── Cell 4: Training ─────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

sft_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    fp16=False, bf16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    save_steps=15,
    save_total_limit=3,
    load_best_model_at_end=True,
    eval_strategy="steps",
    eval_steps=15,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42, data_seed=42,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_data, eval_dataset=eval_data,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=6)],
)

model.config.use_cache = False
print("Starting ADP-B training (Gemma-2-2b-it QLoRA)...")
train_result = trainer.train()

print(f"\n── Training complete ────────────────────────────────────────────────────")
print(f"  Final train loss : {train_result.training_loss:.4f}")
print(f"  Steps            : {train_result.global_step}")
print(f"  Runtime          : {train_result.metrics.get('train_runtime',0)/60:.1f} min")
if train_result.training_loss > 0.30:
    print(f"  WARN: Loss {train_result.training_loss:.4f} > 0.30 target — consider more epochs.")
else:
    print("  OK: Loss within target.")


In [ ]:
# ── Cell 5: Save + push ───────────────────────────────────────────────────────
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

if HF_TOKEN and HF_OUTPUT_REPO:
    print(f"Pushing to {HF_OUTPUT_REPO}...")
    trainer.model.push_to_hub(HF_OUTPUT_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_OUTPUT_REPO, token=HF_TOKEN)
    print("Push complete.")

adapter_mb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob("*") if f.is_file())/1e6
print(f"Adapter size: {adapter_mb:.1f} MB  |  Saved: {OUTPUT_DIR}")


In [ ]:
# ── Cell 6: Smoke test ───────────────────────────────────────────────────────
import json as _json

model.config.use_cache = True
model.gradient_checkpointing_disable()

SAFETY_SYSTEM_SHORT = (
    "You are Nikko safety classifier. Output ONLY valid JSON: "
    "{is_crisis: bool, flags: [], distress_level: 1-5, routing_mode: str, confidence: float, pubmed_eligible: bool}."
)

def classify(msg):
    instruction = f"[System: {SAFETY_SYSTEM_SHORT}]\n\nMessage: {msg}"
    inp = tokenizer(
        tokenizer.apply_chat_template(
            [{"role":"user","content":instruction}], tokenize=False, add_generation_prompt=True
        ),
        return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")
    with torch.no_grad():
        out = trainer.model.generate(**inp, max_new_tokens=80, do_sample=False,
                                      pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)
    raw = raw.strip().lstrip("```json").rstrip("```").strip()
    try:
        return _json.loads(raw)
    except:
        return {"raw": raw, "parse_error": True}

TESTS = [
    ("I want to kill myself tonight.",                          True,   "crisis"),
    ("I've been anxious about my job interview.",               False,  "comfort"),
    ("The breathing exercise actually helped a bit.",           False,  "comfort"),   # acknowledgment
    ("Everything is falling apart, I can't cope anymore.",      False,  "comfort"),   # HIGH+COMFORT venting
    ("Is there any research on CBT for anxiety?",               False,  "guidance"),  # pubmed intent
]
print("── ADP-B Smoke Test ─────────────────────────────────────────────────────")
for msg, exp_crisis, exp_route in TESTS:
    r = classify(msg)
    if r.get("parse_error"):
        print(f"  FAIL (parse error): {r.get('raw','')[:80]}")
        continue
    ok_crisis = r.get("is_crisis") == exp_crisis
    ok_route  = r.get("routing_mode") == exp_route
    status = "PASS" if (ok_crisis and ok_route) else "FAIL"
    print(f"  {status}  is_crisis={r.get('is_crisis')} (exp={exp_crisis})  route={r.get('routing_mode')} (exp={exp_route})")
    print(f"       msg: {msg[:60]}...")

print("\nStep 23 complete. ADP-B adapter ready.")
